# Cursive Transformer — Train & Generate (Colab)

V5 model: RoPE · RMSNorm · SwiGLU · fused FlashAttention · text encoder · classifier-free guidance · BOS + KV-cached generation · structured sampling · **few-shot style cloning** · EMA weights · rendering-aware RLOO.

**Before you start:** `Runtime → Change runtime type → GPU` (an **A100** is ideal).

Steps: 1) check GPU → 2) securely set config → 3) clone + install → 4) warm-started SFT → 5) RLOO pilot → 6) generate.

> Checkpoints now store their architecture and auto-correct mismatched settings at load time,
> so the old `n_layer` mismatch error can't happen with checkpoints trained from this version on.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Configuration

**Rotate the W&B key that appeared in older run configs before continuing.** Get the replacement at https://wandb.ai/authorize. This notebook puts it only in `WANDB_API_KEY`; it is never passed as a CLI argument or uploaded in run config.

`REPO_URL` / `BRANCH` should point at **your** fork containing the modernized code. If you haven't pushed it, use the *Manual upload* cell further down instead.

In [ ]:
REPO_URL      = 'https://github.com/ariedotcodotnz/cursivetransformer.git'  #@param
BRANCH        = 'main'                     #@param

WANDB_ENTITY  = 'cursivetransformer-ng'     #@param
WANDB_PROJECT = 'cursivetransformer-ng'     #@param
SFT_RUN_NAME  = 'modern_v5_sft'             #@param
INIT_FROM_RUN_ID = 'bxf7fmpq'               #@param  (v3 best backbone; warm-start, not resume)

DATASET       = 'bigbank_3500'             #@param  (must be a .json.zip in data/)
NUM_WORDS     = 4                           #@param
MAX_SEQ_LEN   = 1050                        #@param

# --- model architecture (training and inference reuse these same values; the
#     checkpoint also stamps them, so loading auto-corrects any mismatch) ---
N_LAYER         = 5                         #@param
N_EMBD          = 64                        #@param
N_CTX_HEAD      = 4                         #@param
N_CONTEXT_LAYER = 2                         #@param  (bidirectional text-encoder layers; 0 = off)

# --- few-shot style adaptation (style cloning) ---
STYLE_WORDS     = 3                         #@param  (reference words per example; 0 disables style conditioning)
EMA_DECAY       = 0.999                     #@param  (0 disables the averaged eval/sample weights)

# --- training scale ---
BATCH_SIZE    = 64                          #@param  (64 beat 32 in the original repo's A/B)
MAX_STEPS     = 20000                       #@param  (hard cap; deterministic validation can stop earlier)
LOG_EVERY     = 2000                        #@param
EARLY_STOPPING_PATIENCE = 3                 #@param  (validation intervals; 0 disables)
EARLY_STOPPING_MIN_DELTA = 0.001            #@param

import getpass, os
if not os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = getpass.getpass('New/rotated W&B API key: ')
print('config set.')

## 3. Clone the repo + install dependencies

In [ ]:
import os
if not os.path.isdir('cursivetransformer'):
    !git clone -b {BRANCH} {REPO_URL}
%cd cursivetransformer
!pip -q install -r requirements.txt
import wandb
wandb.login()
print('\nDatasets available in data/:')
!ls data/*.json.zip

### (Optional) Manual upload — only if you didn't push to GitHub
Clone the **original** repo, then run this cell and upload your local `model.py`, `data.py`, `train.py`, `sample.py`, `rendering_reward.py`, and `rl_train.py`.

In [ ]:
# from google.colab import files
# uploaded = files.upload()           # select the six files listed above
# for name, content in uploaded.items():
#     with open(name, 'wb') as f: f.write(content)
# print('overwrote:', list(uploaded))

## 4. Stage 1 - warm-started supervised training

Warm-start compatible backbone weights from v3 run `bxf7fmpq`, while initializing the BOS row and style modules normally. This is a new run with a fresh optimizer/schedule. The conservative backbone LR is `1e-3`; the style encoder learns at 3x (`3e-3`).

Every validation pass is deterministic and uses the full test split. W&B logs `test_loss_style_correct`, `test_loss_no_style`, `test_loss_shuffled_style`, and both style gains on the same targets. Correct-style loss should be lowest. The 20k cap and early stopping limit overfitting.

In [ ]:
!python train.py \
  --wandb_entity {WANDB_ENTITY} \
  --wandb_project {WANDB_PROJECT} \
  --wandb_run_name {SFT_RUN_NAME} \
  --dataset_name {DATASET} \
  --num_words {NUM_WORDS} \
  --max_seq_length {MAX_SEQ_LEN} \
  --batch_size {BATCH_SIZE} \
  --n_layer {N_LAYER} \
  --n_embd {N_EMBD} \
  --n_ctx_head {N_CTX_HEAD} \
  --n_context_layer {N_CONTEXT_LAYER} \
  --learning_rate 1e-3 \
  --lr_schedule cosine --warmup_steps 1000 \
  --grad_clip 1.0 \
  --dropout 0.1 \
  --max_steps {MAX_STEPS} \
  --train_size 497000 --test_size 3000 \
  --log_every {LOG_EVERY} \
  --num_workers 8 \
  --downsample_mean 0.65 \
  --cond_drop_prob 0.1 \
  --style_words {STYLE_WORDS} \
  --style_drop_prob 0.1 \
  --ema_decay {EMA_DECAY} \
  --use_bos \
  --init_from_run_id {INIT_FROM_RUN_ID} \
  --style_lr_multiplier 3 \
  --eval_batch_size 100 --eval_max_batches -1 \
  --early_stopping_patience {EARLY_STOPPING_PATIENCE} \
  --early_stopping_min_delta {EARLY_STOPPING_MIN_DELTA} \
  --local_checkpoint_path v5_sft_best.pt \
  --subnetwork_mode full \
  --seed 1337

### Resume a dropped run
If Colab killed your A100 mid-run, set `RESUME_RUN_ID` to the W&B run id and re-run. It restores model + optimizer + scheduler + step. Keep `MAX_STEPS` the same so the cosine schedule lines up.

In [ ]:
RESUME_RUN_ID = ''  #@param
if RESUME_RUN_ID:
    !python train.py \
      --wandb_entity {WANDB_ENTITY} --wandb_project {WANDB_PROJECT} \
      --dataset_name {DATASET} --num_words {NUM_WORDS} --max_seq_length {MAX_SEQ_LEN} \
      --batch_size {BATCH_SIZE} --n_layer {N_LAYER} --n_embd {N_EMBD} --n_ctx_head {N_CTX_HEAD} \
      --n_context_layer {N_CONTEXT_LAYER} --learning_rate 1e-3 \
      --lr_schedule cosine --warmup_steps 1000 --grad_clip 1.0 --dropout 0.1 \
      --max_steps {MAX_STEPS} --train_size 497000 --test_size 3000 --log_every {LOG_EVERY} \
      --num_workers 8 --downsample_mean 0.65 --cond_drop_prob 0.1 --subnetwork_mode full --seed 1337 \
      --style_words {STYLE_WORDS} --style_drop_prob 0.1 --ema_decay {EMA_DECAY} --use_bos \
      --style_lr_multiplier 3 --eval_batch_size 100 --eval_max_batches -1 \
      --early_stopping_patience {EARLY_STOPPING_PATIENCE} --early_stopping_min_delta {EARLY_STOPPING_MIN_DELTA} \
      --local_checkpoint_path v5_sft_best.pt \
      --load_from_run_id {RESUME_RUN_ID}
else:
    print('Set RESUME_RUN_ID above to resume.')

## 5. Stage 2 - rendering-aware RLOO pilot

Paste the completed v5 SFT run id below. This cell uses the W&B API to read the best checkpoint metadata and blocks RL unless the correct style reference beats both no-style and shuffled-style controls. `rl_train.py` then downloads the SFT artifact, loads its EMA policy, and trains from prompt-only BOS rollouts.

The pilot uses four conditions x four rollouts, temperature 1.0, no CFG, two-word prompts, KV-cached sampling, LR `1e-4`, and a `0.25` supervised anchor. Fixed-canvas rendering logs every reward component plus target/reference/best/median/worst candidate grids.

In [ ]:
#@title Verify the SFT style gate through the W&B API
SFT_RUN_ID    = ''                        #@param  (the modern_v5_sft run id)
RLOO_RUN_NAME = 'modern_v5_rloo_pilot'    #@param
RLOO_MAX_UPDATES = 200                    #@param  (do not increase for the first pilot)

if not SFT_RUN_ID:
    raise ValueError('Set SFT_RUN_ID to the completed v5 SFT run.')
api = wandb.Api()
sft_run = api.run(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{SFT_RUN_ID}')
model_artifacts = [a for a in sft_run.logged_artifacts() if a.type == 'model']
if not model_artifacts:
    raise RuntimeError('The SFT run has no model artifact.')
best_artifact = max(model_artifacts, key=lambda a: int(a.name.rsplit(':v', 1)[-1]))
metrics = best_artifact.metadata or {}
correct = metrics.get('test_loss')
no_style = metrics.get('test_loss_no_style')
shuffled = metrics.get('test_loss_shuffled_style')
print('artifact:', best_artifact.name)
print('correct/no-style/shuffled:', correct, no_style, shuffled)
if None in (correct, no_style, shuffled):
    raise RuntimeError('Best artifact is missing deterministic style metrics.')
if not (correct < no_style and correct < shuffled):
    raise RuntimeError('Style gate failed; do not start RLOO from this checkpoint.')
print('SFT style gate passed.')

In [ ]:
!python rl_train.py \
  --sft-run-id {SFT_RUN_ID} \
  --max-updates {RLOO_MAX_UPDATES} \
  --conditions-batch-size 4 --rollouts-per-condition 4 \
  --temperature 1.0 --top-k 0 --top-p 1.0 \
  --rl-learning-rate 1e-4 --rl-warmup-updates 10 --lr-total-updates 1500 \
  --sft-coef 0.25 --grad-clip 1.0 --ema-decay 0.999 \
  --num-words 2 --dataset-name {DATASET} \
  --eval-every 50 --eval-conditions 4 --eval-rollouts 4 \
  --eval-ablation-rollouts 1 --save-every 50 --reward-workers 4 \
  --best-checkpoint-path rl_best_checkpoint.pt \
  --last-checkpoint-path rl_last_checkpoint.pt \
  --wandb-entity {WANDB_ENTITY} --wandb-project {WANDB_PROJECT} \
  --wandb-run-name {RLOO_RUN_NAME} --seed 1337

### Pilot gate and optional continuation

At updates 50/100/150/200, inspect `eval/samples` and the `eval_reward`, `eval_style`, END, grammar, blank, length, and out-of-bounds metrics in W&B. Continue only when held-out reward improves, both style gains stay positive, the visual reward ranking makes sense, SFT loss remains controlled, and failure rates do not rise. The best EMA policy is `rl_best_checkpoint.pt`; resumable state is `rl_last_checkpoint.pt`.

In [ ]:
#@title Continue an approved pilot to 1,500 total updates
RLOO_RUN_ID = ''       #@param  (pilot W&B run id; leave blank to do nothing)
CONTINUE_TO = 1500     #@param  (total updates, not additional updates)
if RLOO_RUN_ID:
    if not os.path.exists('rl_last_checkpoint.pt'):
        raise FileNotFoundError('rl_last_checkpoint.pt is required to continue optimizer/EMA state.')
    !python rl_train.py \
      --sft-run-id {SFT_RUN_ID} \
      --resume-rl-checkpoint rl_last_checkpoint.pt --max-updates {CONTINUE_TO} \
      --conditions-batch-size 4 --rollouts-per-condition 4 \
      --temperature 1.0 --rl-learning-rate 1e-4 --lr-total-updates 1500 --sft-coef 0.25 \
      --num-words 2 --dataset-name {DATASET} \
      --eval-every 50 --eval-conditions 4 --eval-rollouts 4 --eval-ablation-rollouts 1 \
      --save-every 50 --best-checkpoint-path rl_best_checkpoint.pt --last-checkpoint-path rl_last_checkpoint.pt \
      --wandb-entity {WANDB_ENTITY} --wandb-project {WANDB_PROJECT} \
      --wandb-run-id {RLOO_RUN_ID} --wandb-run-name {RLOO_RUN_NAME} --seed 1337
else:
    print('Leave blank after the pilot; set only after the go/no-go review passes.')

## 6. Generate handwriting

The same-runtime default loads local `rl_best_checkpoint.pt`. Otherwise set `RUN_ID` to the RLOO run id; its W&B artifact uses the canonical `best_checkpoint.pt` payload expected by the existing loader. Change `LOCAL_CHECKPOINT_PATH` when sampling an SFT-only checkpoint.

In [ ]:
RUN_ID = ''                              #@param  (RLOO/SFT W&B run id)
LOCAL_CHECKPOINT_PATH = 'rl_best_checkpoint.pt'  #@param

import torch
from model import get_all_args, get_checkpoint
from data import create_datasets
from sample import GenerationParams, generate_paragraph, plot_paragraph

args = get_all_args(use_argparse=False)
args.device          = 'cuda' if torch.cuda.is_available() else 'cpu'
args.dataset_name    = DATASET
args.num_words       = NUM_WORDS
args.max_seq_length  = MAX_SEQ_LEN
args.n_layer         = N_LAYER
args.n_embd          = N_EMBD
args.n_ctx_head      = N_CTX_HEAD
args.n_context_layer = N_CONTEXT_LAYER
args.wandb_entity    = WANDB_ENTITY
args.wandb_project   = WANDB_PROJECT
args.use_bos         = True
args.style_words     = STYLE_WORDS
args.load_from_run_id = RUN_ID
args.local_checkpoint_path = LOCAL_CHECKPOINT_PATH

train_ds, test_ds = create_datasets(args)
args.vocab_size         = train_ds.get_vocab_size()
args.block_size         = train_ds.get_stroke_seq_length()
args.context_block_size = train_ds.get_text_seq_length()
args.context_vocab_size = train_ds.get_char_vocab_size()

model, *_ = get_checkpoint(args, sample_only=True)
model.eval()
print('model loaded.')

In [ ]:
#@title Generate from custom text
TEXT           = 'the quick brown fox jumps over the lazy dog'  #@param
TEMPERATURE    = 0.8   #@param
TOP_P          = 0.95  #@param
GUIDANCE_SCALE = 2.0   #@param  (>1 sharpens spelling/text adherence; try 1.5-2.5)
STRUCTURED     = True  #@param  (forbid tokens that break the stroke pairing)

params = GenerationParams(
    do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
    guidance_scale=GUIDANCE_SCALE, structured=STRUCTURED,
)

offsets = generate_paragraph(model, test_ds, TEXT, params)
fig, ax = plot_paragraph(offsets, TEXT, params=params, include_title=True)
fig

### Fix a misspelled word
Pass the previous `offsets` back in and list the word indices to regenerate (set `show_indices=True` when plotting to read the index numbers).

In [ ]:
REGENERATE_IXS = [2, 4]  #@param
offsets = generate_paragraph(model, test_ds, TEXT, params,
                             word_list_offsets=offsets, regenerate_ixs=REGENERATE_IXS)
fig, ax = plot_paragraph(offsets, TEXT, params=params, show_indices=True, include_title=True)
fig

## 7. Clone someone's handwriting style (few-shot)

Capture a few words of handwriting with `data/collect.html` (or any file in the same JSON format: a list of `{points, metadata}` examples), upload it here, and the model will write **new text in that style**. `STYLE_GUIDANCE` > 1 pushes harder toward the reference style (try 1.5–2.5).

Requires a checkpoint trained with `STYLE_WORDS > 0`.

In [ ]:
#@title Upload a style sample and write in that style
CLONE_TEXT     = 'handwriting cloned from your sample'  #@param
STYLE_GUIDANCE = 2.0   #@param  (>1 mimics the reference style more strongly)

from google.colab import files
from data import load_style_reference

uploaded = files.upload()  # select your style .json / .json.zip
style_path = list(uploaded)[0]
style = load_style_reference(style_path, test_ds)

params = GenerationParams(do_sample=True, temperature=0.8, top_p=0.95,
                          guidance_scale=2.0, structured=True,
                          style_guidance_scale=STYLE_GUIDANCE)
offsets = generate_paragraph(model, test_ds, CLONE_TEXT, params, style=style)
fig, ax = plot_paragraph(offsets, CLONE_TEXT, params=params, include_title=True)
fig